# Trajectory Evals and Statistics

Trajectory evaluation examines the sequence of decisions and tool calls that produced an answer, not only the final text.


## What to learn

- Exact matching works for required tool names and arguments.
- Constraint checks allow different valid paths while forbidding unsafe ones.
- Simulated users test multi-turn behavior but must be calibrated against real interactions.
- Confidence intervals distinguish meaningful changes from sampling noise.

## Decision rule

Score outcomes and trajectories separately. Run each variable case more than once, report sample size and uncertainty, and inspect regressions rather than relying on a single aggregate score.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# Import the dependencies used by this example.
from langsmith import Client

client = Client()

# Define the data shape and small operations before running them.
def exact_tool_path(outputs: dict, reference_outputs: dict) -> dict:
    def tool_names(messages):
        return [call["name"] for message in messages for call in getattr(message, "tool_calls", [])]
    actual = tool_names(outputs["messages"])
    expected = reference_outputs["tool_path"]
    return {"key": "exact_tool_path", "score": int(actual == expected),
            "comment": f"actual={actual}; expected={expected}"}

def target(inputs: dict) -> dict:
    # Execute the configured model or workflow with the input below.
    return agent.invoke({"messages": inputs["messages"]})

experiment = client.evaluate(
    target,
    data="agent-trajectory-regression",
    evaluators=[exact_tool_path],
    experiment_prefix="langgraph-agent-v2",
    max_concurrency=4,
)
experiment


In [ ]:
# Import the dependencies used by this example.
from langsmith import Client
from agentevals.trajectory.match import create_trajectory_match_evaluator

client = Client()
trajectory_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="strict",
)

# Define the data shape and small operations before running them.
def target(inputs: dict) -> dict:
    # Execute the configured model or workflow with the input below.
    return agent.invoke({"messages": inputs["messages"]})

experiment = client.evaluate(
    target,
    data="agent-trajectory-regression",
    evaluators=[trajectory_evaluator],
    experiment_prefix="langgraph-agent-v2",
    max_concurrency=4,
)
experiment


In [ ]:
"""Score simulated agent trajectories against an expected tool-call spec
(in-order matching with an extra-call penalty), then bootstrap a 95% CI on
each variant's per-task pass rate and run a paired permutation test to decide
whether variant B is really better. No API needed -- agent runs are simulated
with seeded randomness; pure Python stdlib throughout."""
# Import the dependencies used by this example.
import random

# ---- 1) Expected-trajectory spec + IN_ORDER matcher with extra penalty ----
EXPECTED = [
    ("search_flights",     {"origin": "SFO", "dest": "JFK"}),
    ("get_reservation",    {"res_id": "R-42"}),
    ("update_reservation", {"res_id": "R-42", "cabin": "economy"}),
    ("send_confirmation",  {"res_id": "R-42"}),
]

# Define the data shape and small operations before running them.
def args_ok(expected_args, actual_args):        # subset matching on args
    return all(actual_args.get(k) == v for k, v in expected_args.items())

def score(actual, expected=EXPECTED, extra_penalty=0.1):
    """Expected calls must appear as an in-order subsequence (IN_ORDER mode);
    every unmatched (extra) actual call costs `extra_penalty`."""
    i = 0
    for name, args in actual:
        if i < len(expected) and name == expected[i][0] and args_ok(expected[i][1], args):
            i += 1
    extras = len(actual) - i
    return max(0.0, i / len(expected) - extra_penalty * extras)

# ---- 2) Simulate two agent variants over shared tasks x k trials ----------
def simulate(rng, p_skip, p_swap, p_extra):
    calls = [(n, dict(a)) for n, a in EXPECTED if rng.random() > p_skip]
    if len(calls) > 1 and rng.random() < p_swap:           # adjacent swap
        j = rng.randrange(len(calls) - 1)
        calls[j], calls[j + 1] = calls[j + 1], calls[j]
    while rng.random() < p_extra:                          # spurious calls
        calls.insert(rng.randrange(len(calls) + 1), ("list_policies", {}))
    return calls

def run(seed, p_skip, p_swap, p_extra, difficulties, k=4):
    rng = random.Random(seed)
    return [[score(simulate(rng, p_skip * d, p_swap * d, p_extra * d)) >= 0.999
             for _ in range(k)] for d in difficulties]

task_rng = random.Random(0)
DIFF = [task_rng.uniform(0.5, 2.0) for _ in range(40)]     # shared per-task difficulty
A = run(seed=1, p_skip=0.06, p_swap=0.10, p_extra=0.25, difficulties=DIFF)  # baseline
B = run(seed=2, p_skip=0.03, p_swap=0.06, p_extra=0.20, difficulties=DIFF)  # new prompt

mean = lambda xs: sum(xs) / len(xs)
ra = [mean(t) for t in A]                                  # per-task avg@k
rb = [mean(t) for t in B]

k = len(A[0])
for name, tr, r in (("A", A, ra), ("B", B, rb)):
    print(f"variant {name}:  avg@{k} = {mean(r):.3f}   "
          f"pass@{k} = {mean([1.0 * any(t) for t in tr]):.3f}   "
          f"pass^{k} = {mean([1.0 * all(t) for t in tr]):.3f}")

# ---- 3) Bootstrap 95% CI on avg@k (resample TASKS, never trials) ----------
def bootstrap_ci(rates, n_boot=4000, seed=7):
    rng = random.Random(seed)
    stats = sorted(mean([rates[rng.randrange(len(rates))] for _ in rates])
                   for _ in range(n_boot))
    return stats[int(0.025 * n_boot)], stats[int(0.975 * n_boot)]

print()
for name, r in (("A", ra), ("B", rb)):
    lo, hi = bootstrap_ci(r)
    print(f"variant {name}:  95% bootstrap CI on avg@{k} = [{lo:.3f}, {hi:.3f}]")

# ---- 4) Paired permutation (sign-flip) test on per-task deltas ------------
def permutation_test(deltas, n_perm=20000, seed=11):
    rng = random.Random(seed)
    obs = mean(deltas)
    hits = sum(abs(mean([x if rng.random() < 0.5 else -x for x in deltas])) >= abs(obs)
               for _ in range(n_perm))
    return obs, (hits + 1) / (n_perm + 1)                  # add-one smoothing

deltas = [b - a for a, b in zip(ra, rb)]
obs, p = permutation_test(deltas)
print(f"\npaired mean delta (B - A) = {obs:+.3f}   permutation p = {p:.4f}")
print("verdict:", "B is credibly better -- ship it" if p < 0.05 else
      "difference is within noise -- do NOT ship on this evidence")


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
